In [1]:
import pandas as pd

In [2]:
pd.set_option('display.max_columns', None)

In [32]:
"""
Baseline Balance Diagnostic — W1 vs W4
==========================================================================
Purpose:
  Test whether W1 (Jul 10-13, pre-experiment) and W4 (Jul 28-Aug 10,
  mid-experiment washout) are comparable baselines.

  If drivers are permanently shifted by W2/W3 exposure, W4 productivity
  will be higher than W1 even with no active incentive — making W4 a
  "dirty" control period that biases DiD coefficients downward.

Tests run:
  1. Raw mean comparison  — W1 vs W4 per outcome, overall and per zone
  2. Driver-level paired test — restrict to drivers present in BOTH W1 and W4,
     compare each driver's W1 mean vs W4 mean (within-driver shift)
  3. Distribution shift — decile breakdown to check if the shift is
     concentrated at the top (high-effort drivers responding to past incentives)
  4. Zone-type breakdown — win / lose / swing zones separately
     (anticipation effects may be stronger in contested zones)
  5. Event-study plot data — daily mean output W1 through W4 to visualise
     any trend or level-shift at the W4 re-entry point

Outputs:
  balance_summary.csv          — mean / SD / N per week × outcome × zone_type
  balance_paired.csv           — within-driver W1→W4 shift, t-test results
  balance_deciles.csv          — decile means W1 vs W4
  balance_daily_means.csv      — daily mean for event-study plotting
  balance_report.txt           — readable summary of all findings
"""

import os
import re
import glob
import warnings
import pandas as pd
import numpy as np
from scipy import stats

warnings.filterwarnings("ignore")

# ── Config (mirror your main script) ─────────────────────────────────────────
DATA_DIR   = "../mainexperiment/weekly/tier_merged_updated/"
OUTPUT_DIR = "./results"

OUTCOMES      = ["completed", "total_online_hour"]
TREATMENT_COL = "Incentive"
DRIVER_ID_COL = "driver_id"
DATE_COL      = "date"
ZONE_TYPE_COL = "status"
REFERENCE_LEVEL = "No Incentive"

WEEK_MAP = [
    ("0710_0713", "W1_Jul10"),
    ("0714_0720", "W2_Jul14"),
    ("0721_0727", "W3_Jul21"),
    ("0728_0810", "W4_Jul28"),
    ("0811_0815", "W5_Aug11"),
    ("0816_0822", "W6_Aug16"),
    ("0823_0829", "W7_Aug23"),
]

W1_LABEL = "W1_Jul10"
W4_LABEL = "W4_Jul28"

# ── Helpers ───────────────────────────────────────────────────────────────────
def safe(c):
    return re.sub(r"[^0-9a-zA-Z_]", "_", str(c))

def load_files(data_dir):
    pattern = os.path.join(data_dir, "tiers_merged_all-*.csv")
    files   = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files in: {os.path.abspath(data_dir)}")
    dfs = []
    for fp in files:
        fname      = os.path.basename(fp)
        fname_norm = fname.replace("-", "_")
        week_label = next((lbl for pat, lbl in WEEK_MAP if pat in fname_norm), fname)
        df         = pd.read_csv(fp, low_memory=False)
        df["_week"] = week_label
        dfs.append(df)
    out = pd.concat(dfs, ignore_index=True)
    return out

def cohens_d(a, b):
    """Pooled Cohen's d."""
    na, nb = len(a), len(b)
    if na < 2 or nb < 2:
        return np.nan
    pooled_sd = np.sqrt(((na - 1) * a.std()**2 + (nb - 1) * b.std()**2) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled_sd if pooled_sd > 0 else np.nan

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    report_lines = []

    def rpt(line=""):
        print(line)
        report_lines.append(line)

    rpt("=" * 72)
    rpt("Baseline Balance Diagnostic — W1 (pre-experiment) vs W4 (mid-experiment)")
    rpt("=" * 72)
    rpt()
    rpt("Hypothesis: If prior incentive exposure (W2, W3) permanently shifts")
    rpt("driver effort, W4 means will be HIGHER than W1 means even with no")
    rpt("active incentive. This would make W4 a contaminated control period")
    rpt("and bias all DiD coefficients DOWNWARD (incentive effects look smaller")
    rpt("or even negative because the baseline is already elevated).")
    rpt()

    df = load_files(DATA_DIR)
    df = df.loc[:, ~df.columns.duplicated()]

    # Rename to safe column names
    rename = {c: safe(c) for c in
              OUTCOMES + [TREATMENT_COL, DRIVER_ID_COL, DATE_COL, ZONE_TYPE_COL, "_week"]
              if c and c in df.columns}
    df = df.rename(columns=rename)

    treatment_s = safe(TREATMENT_COL)
    driver_s    = safe(DRIVER_ID_COL)
    date_s      = safe(DATE_COL)
    zone_s      = safe(ZONE_TYPE_COL)
    week_s      = "_week"

    for c in OUTCOMES:
        df[safe(c)] = pd.to_numeric(df[safe(c)], errors="coerce")

    # Restrict to baseline weeks only for most tests
    baseline = df[df[week_s].isin([W1_LABEL, W4_LABEL])].copy()
    baseline = baseline[baseline[treatment_s] == REFERENCE_LEVEL].copy()

    rpt(f"Baseline rows  — W1: {(baseline[week_s]==W1_LABEL).sum():,}  "
        f"W4: {(baseline[week_s]==W4_LABEL).sum():,}")
    rpt(f"Unique drivers — W1: {baseline[baseline[week_s]==W1_LABEL][driver_s].nunique():,}  "
        f"W4: {baseline[baseline[week_s]==W4_LABEL][driver_s].nunique():,}")
    rpt()

    summary_rows  = []
    paired_rows   = []
    decile_rows   = []
    daily_rows    = []

    for outcome_raw in OUTCOMES:
        outcome_s = safe(outcome_raw)
        if outcome_s not in df.columns:
            rpt(f"  ⚠ {outcome_raw} not found — skipping")
            continue

        rpt("─" * 72)
        rpt(f"OUTCOME: {outcome_raw}")
        rpt("─" * 72)

        w1 = baseline[baseline[week_s] == W1_LABEL][outcome_s].dropna()
        w4 = baseline[baseline[week_s] == W4_LABEL][outcome_s].dropna()

        # ── Test 1: Raw means ─────────────────────────────────────────────
        rpt()
        rpt("TEST 1 — Raw mean comparison (all baseline driver-days)")
        rpt(f"  W1 mean : {w1.mean():.4f}  SD={w1.std():.4f}  n={len(w1):,}")
        rpt(f"  W4 mean : {w4.mean():.4f}  SD={w4.std():.4f}  n={len(w4):,}")

        t_stat, p_val = stats.ttest_ind(w4, w1, equal_var=False)
        d             = cohens_d(w4, w1)
        pct_diff      = (w4.mean() - w1.mean()) / w1.mean() * 100 if w1.mean() != 0 else np.nan

        rpt(f"  W4 vs W1: Δ={w4.mean()-w1.mean():+.4f}  ({pct_diff:+.1f}%)  "
            f"t={t_stat:.3f}  p={p_val:.4f}  Cohen's d={d:.3f}")
        if p_val < 0.05 and w4.mean() > w1.mean():
            rpt(f"  ⚠ W4 is SIGNIFICANTLY HIGHER than W1 — baseline contamination likely")
        elif p_val < 0.05 and w4.mean() < w1.mean():
            rpt(f"  ⚠ W4 is SIGNIFICANTLY LOWER than W1 — seasonal/demand difference")
        else:
            rpt(f"  ✓ No significant difference in raw means")

        summary_rows.append({
            "outcome": outcome_raw, "week": "W1", "zone_type": "ALL",
            "mean": round(w1.mean(), 4), "sd": round(w1.std(), 4), "n": len(w1),
        })
        summary_rows.append({
            "outcome": outcome_raw, "week": "W4", "zone_type": "ALL",
            "mean": round(w4.mean(), 4), "sd": round(w4.std(), 4), "n": len(w4),
            "delta_vs_W1": round(w4.mean() - w1.mean(), 4),
            "pct_diff": round(pct_diff, 2),
            "tstat": round(t_stat, 4), "pval": round(p_val, 4),
            "cohens_d": round(d, 4),
        })

        # ── Test 2: Within-driver paired comparison ───────────────────────
        rpt()
        rpt("TEST 2 — Within-driver paired comparison")
        rpt("  (drivers present in BOTH W1 and W4 only)")

        # Driver-level daily means
        drv_means = (
            baseline[baseline[outcome_s].notna()]
            .groupby([driver_s, week_s])[outcome_s]
            .mean()
            .unstack(week_s)
        )
        both = drv_means.dropna(subset=[W1_LABEL, W4_LABEL])
        rpt(f"  Drivers in both W1 & W4: {len(both):,}")

        if len(both) > 1:
            w1_paired = both[W1_LABEL]
            w4_paired = both[W4_LABEL]
            diff      = w4_paired - w1_paired

            t_stat_p, p_val_p = stats.ttest_rel(w4_paired, w1_paired)
            d_paired          = cohens_d(w4_paired, w1_paired)
            pct_p             = diff.mean() / w1_paired.mean() * 100 if w1_paired.mean() != 0 else np.nan

            rpt(f"  Mean W1 (driver avg) : {w1_paired.mean():.4f}")
            rpt(f"  Mean W4 (driver avg) : {w4_paired.mean():.4f}")
            rpt(f"  Mean within-driver Δ : {diff.mean():+.4f}  ({pct_p:+.1f}%)")
            rpt(f"  Paired t-test        : t={t_stat_p:.3f}  p={p_val_p:.4f}  Cohen's d={d_paired:.3f}")

            pct_positive = (diff > 0).mean() * 100
            rpt(f"  Drivers with W4 > W1 : {pct_positive:.1f}%  "
                f"(vs {100-pct_positive:.1f}% with W4 ≤ W1)")

            if p_val_p < 0.05 and diff.mean() > 0:
                rpt(f"  ⚠ SAME drivers are MORE productive in W4 than W1 — "
                    f"consistent with prior-incentive habituation / ratchet effect")
            elif p_val_p < 0.05 and diff.mean() < 0:
                rpt(f"  ⚠ SAME drivers are LESS productive in W4 — "
                    f"possible fatigue / demand seasonality")
            else:
                rpt(f"  ✓ No significant within-driver shift W1→W4")

            paired_rows.append({
                "outcome": outcome_raw,
                "n_drivers": len(both),
                "mean_W1": round(w1_paired.mean(), 4),
                "mean_W4": round(w4_paired.mean(), 4),
                "mean_delta": round(diff.mean(), 4),
                "pct_diff": round(pct_p, 2),
                "pct_drivers_higher_in_W4": round(pct_positive, 1),
                "tstat_paired": round(t_stat_p, 4),
                "pval_paired": round(p_val_p, 4),
                "cohens_d": round(d_paired, 4),
            })

        # ── Test 3: Decile breakdown ──────────────────────────────────────
        rpt()
        rpt("TEST 3 — Decile breakdown (where is the shift concentrated?)")
        rpt("  A top-decile shift → high-effort drivers responding to past incentives")
        rpt("  A uniform shift    → platform/demand-level change, not behavioural")

        # Use W1 decile cutoffs to assign all observations to deciles
        w1_vals = baseline[baseline[week_s] == W1_LABEL][outcome_s].dropna()
        decile_edges = np.nanpercentile(w1_vals, np.arange(0, 110, 10))
        decile_edges[-1] += 1e-9  # ensure max value included

        for wk_label, wk_rows in [(W1_LABEL, w1_vals),
                                   (W4_LABEL, baseline[baseline[week_s]==W4_LABEL][outcome_s].dropna())]:
            decile_idx = np.digitize(wk_rows, decile_edges[:-1]) - 1
            decile_idx = np.clip(decile_idx, 0, 9)
            for d_idx in range(10):
                vals = wk_rows.values[decile_idx == d_idx]
                decile_rows.append({
                    "outcome": outcome_raw,
                    "week": wk_label,
                    "decile": d_idx + 1,
                    "n": len(vals),
                    "mean": round(np.nanmean(vals), 4) if len(vals) > 0 else np.nan,
                })

        dec_df = pd.DataFrame(decile_rows)
        dec_df = dec_df[dec_df["outcome"] == outcome_raw]
        if not dec_df.empty:
            pivot_dec = dec_df.pivot_table(
                index="decile", columns="week", values="mean"
            ).reindex(columns=[W1_LABEL, W4_LABEL])
            if W1_LABEL in pivot_dec.columns and W4_LABEL in pivot_dec.columns:
                pivot_dec["delta"] = pivot_dec[W4_LABEL] - pivot_dec[W1_LABEL]
                pivot_dec["pct"]   = (pivot_dec["delta"] / pivot_dec[W1_LABEL] * 100).round(1)
                rpt(pivot_dec.round(4).to_string())

        # ── Test 4: Zone-type breakdown ───────────────────────────────────
        if zone_s in baseline.columns:
            rpt()
            rpt("TEST 4 — Zone-type breakdown (Win / Lose / Swing)")
            zone_types = sorted(baseline[zone_s].dropna().unique())
            for zt in zone_types:
                sub = baseline[baseline[zone_s] == zt]
                z_w1 = sub[sub[week_s] == W1_LABEL][outcome_s].dropna()
                z_w4 = sub[sub[week_s] == W4_LABEL][outcome_s].dropna()
                if len(z_w1) < 5 or len(z_w4) < 5:
                    continue
                z_t, z_p = stats.ttest_ind(z_w4, z_w1, equal_var=False)
                z_d      = cohens_d(z_w4, z_w1)
                z_pct    = (z_w4.mean() - z_w1.mean()) / z_w1.mean() * 100 if z_w1.mean() != 0 else np.nan
                flag     = "⚠" if z_p < 0.05 else "✓"
                rpt(f"  {flag} {zt:<8}  W1={z_w1.mean():.3f}  W4={z_w4.mean():.3f}  "
                    f"Δ={z_w4.mean()-z_w1.mean():+.3f} ({z_pct:+.1f}%)  "
                    f"p={z_p:.4f}  d={z_d:.3f}")

                summary_rows.append({
                    "outcome": outcome_raw, "week": "W4_vs_W1", "zone_type": zt,
                    "mean": round(z_w4.mean(), 4), "sd": round(z_w4.std(), 4),
                    "n": len(z_w4),
                    "delta_vs_W1": round(z_w4.mean() - z_w1.mean(), 4),
                    "pct_diff": round(z_pct, 2),
                    "tstat": round(z_t, 4), "pval": round(z_p, 4),
                    "cohens_d": round(z_d, 4),
                })

        # ── Test 5: Daily means for event-study plot ──────────────────────
        rpt()
        rpt("TEST 5 — Daily means across all baseline + incentive weeks")
        rpt("  (use balance_daily_means.csv to plot an event-study)")

        daily = (
            df[df[outcome_s].notna() & df[treatment_s].notna()]
            .groupby([date_s, week_s, treatment_s])[outcome_s]
            .agg(["mean", "count"])
            .reset_index()
            .rename(columns={"mean": f"mean_{outcome_s}", "count": "n"})
        )
        daily["outcome"] = outcome_raw
        daily_rows.append(daily)

        rpt()

    # ── Save outputs ──────────────────────────────────────────────────────────
    pd.DataFrame(summary_rows).round(4).to_csv(
        os.path.join(OUTPUT_DIR, "balance_summary.csv"), index=False)
    print(f"  ✓ balance_summary.csv")

    if paired_rows:
        pd.DataFrame(paired_rows).round(4).to_csv(
            os.path.join(OUTPUT_DIR, "balance_paired.csv"), index=False)
        print(f"  ✓ balance_paired.csv")

    if decile_rows:
        pd.DataFrame(decile_rows).round(4).to_csv(
            os.path.join(OUTPUT_DIR, "balance_deciles.csv"), index=False)
        print(f"  ✓ balance_deciles.csv")

    if daily_rows:
        pd.concat(daily_rows, ignore_index=True).to_csv(
            os.path.join(OUTPUT_DIR, "balance_daily_means.csv"), index=False)
        print(f"  ✓ balance_daily_means.csv")

    # ── Interpretation guide ──────────────────────────────────────────────────
    rpt()
    rpt("=" * 72)
    rpt("INTERPRETATION GUIDE")
    rpt("=" * 72)
    rpt()
    rpt("If W4 > W1 (Tests 1 & 2, p < 0.05):")
    rpt("  → W4 is a contaminated baseline. Using it as a control period")
    rpt("    inflates the no-incentive mean, so DiD β = (incentive - baseline)")
    rpt("    is biased DOWNWARD — incentive effects look smaller / negative.")
    rpt()
    rpt("What to do if contamination is confirmed:")
    rpt("  Option A — Drop W4 from baseline, use W1 only.")
    rpt("    Pro: clean pre-experiment baseline.")
    rpt("    Con: only 4 days of baseline, noisier estimates.")
    rpt()
    rpt("  Option B — Use W4 as a 'washout' week and exclude it entirely")
    rpt("    (neither baseline nor treatment). Restrict baseline to W1.")
    rpt("    This is the most conservative / defensible choice.")
    rpt()
    rpt("  Option C — Interact baseline week (W1 vs W4) as a covariate.")
    rpt("    Allows the model to absorb the W1→W4 level shift explicitly,")
    rpt("    but requires the shift to be homogeneous across drivers.")
    rpt()
    rpt("  Option D — Use a pre/post design with W1 as the only baseline")
    rpt("    and treat each incentive week separately (W2, W3, W5, W6, W7).")
    rpt("    Day FE still absorbs day-of-week; driver FE handles selection.")
    rpt()

    txt_path = os.path.join(OUTPUT_DIR, "balance_report.txt")
    with open(txt_path, "w") as f:
        f.write("\n".join(report_lines))
    print(f"  ✓ balance_report.txt")
    print()
    print("Done. Run `balance_daily_means.csv` through your plotting tool")
    print("to visualise the daily trend across W1→W4 — a visible level-shift")
    print("at the W4 boundary (compared to W1) confirms contamination.")

if __name__ == "__main__":
    main()

Baseline Balance Diagnostic — W1 (pre-experiment) vs W4 (mid-experiment)

Hypothesis: If prior incentive exposure (W2, W3) permanently shifts
driver effort, W4 means will be HIGHER than W1 means even with no
active incentive. This would make W4 a contaminated control period
and bias all DiD coefficients DOWNWARD (incentive effects look smaller
or even negative because the baseline is already elevated).

Baseline rows  — W1: 15,566  W4: 63,437
Unique drivers — W1: 5,208  W4: 7,308

────────────────────────────────────────────────────────────────────────
OUTCOME: completed
────────────────────────────────────────────────────────────────────────

TEST 1 — Raw mean comparison (all baseline driver-days)
  W1 mean : 23.1932  SD=14.8915  n=15,566
  W4 mean : 21.9348  SD=14.6645  n=63,437
  W4 vs W1: Δ=-1.2585  (-5.4%)  t=-9.476  p=0.0000  Cohen's d=-0.086
  ⚠ W4 is SIGNIFICANTLY LOWER than W1 — seasonal/demand difference

TEST 2 — Within-driver paired comparison
  (drivers present in BOTH W1 

In [30]:
"""
Difference-in-Differences Panel Regression — Day Level
==========================================================================
Design:
  - Same experiment as weekly analysis but at driver × day granularity
  - W1 (Jul 10-13) + W4 (Jul 28-Aug 10) = no-incentive baseline days
  - W2, W3, W5, W6, W7 = incentive days

Model:
  y_it = α_i + δ_t + β·Incentive_it + ε_it

  Where:
    y_it  = completed orders OR hours worked for driver i on day t
    α_i   = driver fixed effects  (each driver's baseline productivity)
    δ_t   = day fixed effects     (absorbs all day-level shocks —
                                   demand, weather, day-of-week, holidays)
    β     = causal effect of each incentive vs no-incentive days
    SE    = clustered by zone

  Two-Way Fixed Effects (TWFE) at the day level:
  "How many MORE orders / hours did a driver log on days they had
   incentive X compared to their own no-incentive baseline days?"

  Advantages over weekly model:
  - ~6x more observations (driver-days vs driver-weeks)
  - Day FE absorbs day-of-week effects, holidays, platform events
  - Can detect within-week variation in incentive response
  - No need to normalise by days (outcome is already per-day)

BASELINE WEIGHTING FIX (v2):
  W1 = 4 days (Jul 10-13), W4 = 14 days (Jul 28-Aug 10).
  Without correction W4 contributes 3.5× more driver-days to the baseline
  pool, inflating the no-incentive mean and biasing DiD coefficients downward.

  Fix: every no-incentive W4 row is assigned weight w = 4/14 ≈ 0.286.
  Every W1 no-incentive row and all incentive rows keep weight w = 1.0.
  This makes W1 and W4 contribute equally to the baseline in expectation,
  while preserving all observations (no information is discarded).

  Weights are passed to statsmodels via freq_weights in all three OLS
  models (TWFE, Pooled OLS, per-day TWFE) so the fix is consistent.

Outputs (one set per outcome):
  did_day_results_{outcome}.csv     — tidy results all models
  did_day_pivot_{outcome}.csv       — coef matrix incentive × week
  did_day_summary_{outcome}.txt     — regression table
  did_day_zone_{outcome}.csv        — zone interaction results
  did_day_zone_marginal_{outcome}.csv — total effects per zone
"""

import os
import re
import glob
import warnings
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════

DATA_DIR   = "../mainexperiment/weekly/tier_merged_updated/"
OUTPUT_DIR = "./results"

# Both outcomes will be run in sequence
OUTCOMES = [
    "completed",          # completed orders per day
    "total_online_hour",  # hours worked per day
]

TREATMENT_COL = "Incentive"
DRIVER_ID_COL = "driver_id"
DATE_COL      = "date"        # date column in raw data — update if named differently

ZONE_TYPE_COL = "status"      # win / lose / swing — actual column name in your data
CLUSTER_BY    = "zone"        # update this if your zone ID column has a different name

REFERENCE_LEVEL = "No Incentive"

# ── Baseline week lengths (used for W4 normalisation) ─────────────────────
# W1 = 4 days (Jul 10-13), W4 = 14 days (Jul 28-Aug 10).
# All no-incentive rows whose _week == W4_WEEK_LABEL get weight W1_DAYS/W4_DAYS.
W1_WEEK_LABEL = "W1_Jul10"
W4_WEEK_LABEL = "W4_Jul28"
W1_DAYS       = 4
W4_DAYS       = 14
W4_WEIGHT     = W1_DAYS / W4_DAYS   # ≈ 0.2857  (deterministic, reproducible)

# Week labels — used to add a week grouping column for per-week models
WEEK_MAP = [
    ("0710_0713", "W1_Jul10"),
    ("0714_0720", "W2_Jul14"),
    ("0721_0727", "W3_Jul21"),
    ("0728_0810", "W4_Jul28"),
    ("0811_0815", "W5_Aug11"),
    ("0816_0822", "W6_Aug16"),
    ("0823_0829", "W7_Aug23"),
]

# ═══════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════

def safe(c):
    return re.sub(r"[^0-9a-zA-Z_]", "_", str(c))

def sig(p):
    if p < 0.01:  return "***"
    if p < 0.05:  return "**"
    if p < 0.10:  return "*"
    return ""

def load_files(data_dir):
    pattern = os.path.join(data_dir, "tiers_merged_all-*.csv")
    files   = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No files matching 'tiers_merged_all-*.csv' in: {os.path.abspath(data_dir)}"
        )
    dfs = []
    for fp in files:
        fname      = os.path.basename(fp)
        fname_norm = fname.replace("-", "_")
        week_label = next((lbl for pat, lbl in WEEK_MAP if pat in fname_norm), fname)
        df         = pd.read_csv(fp, low_memory=False)
        df["_week"] = week_label
        dfs.append(df)
        print(f"  {fname:<45} → {week_label}  ({len(df):,} rows)")
    out = pd.concat(dfs, ignore_index=True)
    print(f"\n  Total rows: {len(out):,}")
    return out


def weight_baseline(panel, treatment_s, week_s, ref):
    """
    Assign observation weights to correct for unequal baseline week lengths.

    W1 has 4 days; W4 has 14 days.  Without weighting, W4 contributes 3.5×
    more no-incentive driver-days to the baseline pool, which inflates the
    reference mean and biases DiD coefficients downward.

    Rule (deterministic / reproducible):
      • No-incentive row in W4  → weight = W1_DAYS / W4_DAYS  (≈ 0.2857)
      • All other rows          → weight = 1.0

    The weights are used as freq_weights in every OLS call so the fix is
    consistent across Model 1 (TWFE pooled), Model 2 (Pooled OLS), and
    Model 3 (per-day TWFE).

    Note: freq_weights in statsmodels scales the score contributions
    (equivalent to replicating rows fractionally), which is the correct
    treatment for rebalancing rather than precision weighting.
    """
    is_baseline_w4 = (
        (panel[treatment_s] == ref) &
        (panel[week_s] == W4_WEEK_LABEL)
    )
    panel = panel.copy()
    panel["_weight"] = np.where(is_baseline_w4, W4_WEIGHT, 1.0)

    n_w1 = ((panel[treatment_s] == ref) & (panel[week_s] == W1_WEEK_LABEL)).sum()
    n_w4 = is_baseline_w4.sum()
    eff_w4 = n_w4 * W4_WEIGHT

    print(f"\n  Baseline weighting (W4 normalisation):")
    print(f"    W1 no-incentive rows : {n_w1:,}  (weight = 1.000)")
    print(f"    W4 no-incentive rows : {n_w4:,}  → effective {eff_w4:.1f}  "
          f"(weight = {W4_WEIGHT:.4f} = {W1_DAYS}/{W4_DAYS})")
    print(f"    All other rows       : weight = 1.000")

    return panel


def build_panel(df, outcome_s, treatment_s, driver_s, date_s,
                cluster_s, zone_type_s, week_s):
    """
    Build a driver × day panel.

    If there is already one row per driver-day, this just selects and
    cleans the relevant columns. If there are multiple rows per driver-day
    (sub-day records), it aggregates by summing the outcome and taking
    the mode of categoricals.

    Returns a clean panel with one row per driver × day.
    """
    # Drop duplicate columns before anything else
    df = df.loc[:, ~df.columns.duplicated()]

    needed = [c for c in [outcome_s, treatment_s, driver_s, date_s,
                           cluster_s, zone_type_s, week_s] if c and c in df.columns]
    panel  = df[needed].copy()
    panel[outcome_s] = pd.to_numeric(panel[outcome_s], errors="coerce")

    # Check if already one row per driver-day
    n_raw    = len(panel)
    n_unique = panel.groupby([driver_s, date_s]).ngroups

    if n_raw == n_unique:
        print(f"  Already one row per driver-day ({n_raw:,} rows)")
        panel = panel.dropna(subset=[outcome_s, treatment_s])
        return panel

    # Aggregate to driver-day
    print(f"  Multiple rows per driver-day detected — aggregating...")
    cat_cols = [c for c in [treatment_s, cluster_s, zone_type_s, week_s]
                if c and c in panel.columns]
    agg_dict = {outcome_s: "sum"}
    for c in cat_cols:
        agg_dict[c] = lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan

    panel = panel.groupby([driver_s, date_s], as_index=False).agg(agg_dict)
    panel = panel.dropna(subset=[outcome_s, treatment_s])
    print(f"  After aggregation: {len(panel):,} driver-day rows")
    return panel

def demean_within_driver(panel, outcome_s, driver_s, weight_col="_weight"):
    """
    Within-transformation: subtract each driver's WEIGHTED mean (= driver FE).

    Using the observation weights when computing the driver mean ensures the
    demeaning is consistent with the weighted OLS estimator.
    """
    dm_col = outcome_s + "_dm"

    if weight_col in panel.columns:
        # Weighted driver mean: sum(w * y) / sum(w)
        w  = panel[weight_col]
        wy = w * panel[outcome_s]
        panel[dm_col] = (
            panel[outcome_s]
            - wy.groupby(panel[driver_s]).transform("sum")
              / w.groupby(panel[driver_s]).transform("sum")
            + (wy.sum() / w.sum())   # add back grand weighted mean to keep scale
        )
    else:
        panel[dm_col] = (
            panel[outcome_s]
            - panel.groupby(driver_s)[outcome_s].transform("mean")
            + panel[outcome_s].mean()
        )
    return panel, dm_col

def run_ols_clustered(formula, data, cluster_col, weight_col="_weight"):
    """
    Fit WLS with freq_weights when a weight column is present, then apply
    clustered SEs.  Falls back to HC3 if clustering fails.
    """
    data = data.loc[:, ~data.columns.duplicated()]

    # Build freq_weights vector (None → unweighted)
    freq_w = None
    if weight_col and weight_col in data.columns:
        freq_w = data[weight_col].values

    try:
        model = smf.wls(formula, data=data, weights=freq_w) if freq_w is not None \
                else smf.ols(formula, data=data)

        if cluster_col and isinstance(cluster_col, str) and cluster_col in data.columns:
            col = data[cluster_col]
            if isinstance(col, pd.Series) and int(col.nunique()) > 1:
                return model.fit(
                    cov_type="cluster",
                    cov_kwds={"groups": col},
                )
        return model.fit(cov_type="HC3")

    except Exception as e:
        print(f"  Warning: weighted/clustered fit failed ({e}) - falling back to OLS/HC3")
        return smf.ols(formula, data=data).fit(cov_type="HC3")

def extract_incentive_coefs(model, treatment_s, ref):
    rows = []
    for param in model.params.index:
        if treatment_s in param and "Intercept" not in param:
            match = re.search(r"\[T\.(.*?)\]", param)
            label = match.group(1) if match else param
            ci    = model.conf_int().loc[param]
            rows.append({
                "incentive": label,
                "coef":    round(model.params[param],  4),
                "se":      round(model.bse[param],     4),
                "tstat":   round(model.tvalues[param], 4),
                "pval":    round(model.pvalues[param], 4),
                "ci_low":  round(ci[0], 4),
                "ci_high": round(ci[1], 4),
            })
    return rows

def print_coef_table(rows, ref, outcome_label):
    unit = "orders" if "completed" in outcome_label else "hours"
    print(f"\n  {'Incentive':<38} {'Coef':>9} {'SE':>7} {'p':>7}  Sig  Interpretation")
    print("  " + "─" * 88)
    for r in sorted(rows, key=lambda x: -x["coef"]):
        direction = (f"+{r['coef']:.4f} {unit}/day vs {ref}"
                     if r["coef"] >= 0
                     else f"{r['coef']:.4f} {unit}/day vs {ref}")
        print(f"  {r['incentive']:<38} {r['coef']:>9.4f} {r['se']:>7.4f}"
              f" {r['pval']:>7.3f}  {sig(r['pval']):<3}  {direction}")

def extract_interaction_coefs(model, treatment_s, zone_type_s):
    rows = []
    ci_df = model.conf_int()
    for param in model.params.index:
        if "Intercept" in param:
            continue
        has_incentive = treatment_s  in param
        has_zone      = zone_type_s  in param
        if not has_incentive:
            continue
        inc_match  = re.search(rf"{treatment_s}\[T\.(.*?)\]",  param)
        zone_match = re.search(rf"{zone_type_s}\[T\.(.*?)\]",  param)
        incentive_label = inc_match.group(1)  if inc_match  else param
        zone_label      = zone_match.group(1) if zone_match else "reference_zone"
        ci = ci_df.loc[param]
        rows.append({
            "incentive": incentive_label,
            "zone_type": zone_label,
            "term":      "interaction" if has_zone else "main_effect",
            "coef":      round(model.params[param],  4),
            "se":        round(model.bse[param],     4),
            "tstat":     round(model.tvalues[param], 4),
            "pval":      round(model.pvalues[param], 4),
            "ci_low":    round(ci[0], 4),
            "ci_high":   round(ci[1], 4),
        })
    return rows

def compute_zone_marginal_effects(interaction_rows, zone_types, ref_zone):
    df     = pd.DataFrame(interaction_rows)
    main   = df[df["term"] == "main_effect"].set_index("incentive")
    inter  = df[df["term"] == "interaction"].set_index(["incentive", "zone_type"])
    results = []
    for inc in main.index.unique():
        m = main.loc[inc]
        results.append({
            "incentive":   inc, "zone_type": ref_zone,
            "total_coef":  m["coef"], "pval": m["pval"],
            "note":        "main effect (reference zone)"
        })
        for zone in zone_types:
            if zone == ref_zone:
                continue
            if (inc, zone) in inter.index:
                i     = inter.loc[(inc, zone)]
                total = m["coef"] + i["coef"]
                results.append({
                    "incentive":          inc,
                    "zone_type":          zone,
                    "total_coef":         round(total, 4),
                    "interaction_coef":   i["coef"],
                    "interaction_pval":   i["pval"],
                    "pval":               i["pval"],
                    "note":               "main + interaction"
                })
            else:
                results.append({
                    "incentive":  inc, "zone_type": zone,
                    "total_coef": m["coef"], "pval": np.nan,
                    "note":       "no interaction term found"
                })
    return pd.DataFrame(results)

# ═══════════════════════════════════════════════════════
# PER-OUTCOME REGRESSION RUNNER
# ═══════════════════════════════════════════════════════

def run_outcome(df, outcome_raw, treatment_s, driver_s, date_s,
                cluster_s, zone_type_s, week_s, ref, output_dir):

    outcome_s     = safe(outcome_raw)
    outcome_label = outcome_raw

    print(f"\n{'═'*70}")
    print(f"OUTCOME: {outcome_raw}")
    print(f"{'═'*70}")

    # ── Build panel ───────────────────────────────────
    panel = build_panel(df, outcome_s, treatment_s, driver_s, date_s,
                        cluster_s, zone_type_s, week_s)

    # ── Apply baseline weighting (W4 normalisation) ───
    panel = weight_baseline(panel, treatment_s, week_s, ref)

    print(f"\n  Unique drivers: {panel[driver_s].nunique():,}")
    print(f"  Unique days:    {panel[date_s].nunique():,}")
    print(f"  Total obs:      {len(panel):,}")

    print(f"\n  Mean {outcome_raw} by incentive (raw, unadjusted):")
    means = panel.groupby(treatment_s)[outcome_s].agg(["mean","count"]).round(3)
    means.columns = [f"mean_{outcome_s}", "n_driver_days"]
    print(means.to_string())

    # ── Driver FE via weighted within-demeaning ───────
    panel, dm_col = demean_within_driver(panel, outcome_s, driver_s, weight_col="_weight")

    treatment_expr = f"C({treatment_s}, Treatment('{ref}'))"
    day_fe_expr    = f"C({date_s})"

    # ══════════════════════════════════════════════════
    # MODEL 1 — TWFE (driver FE + day FE), weighted
    # ══════════════════════════════════════════════════
    print(f"\n{'─'*70}")
    print(f"MODEL 1: TWFE  (driver FE via weighted demeaning + day FE, W4-normalised)")
    print(f"  {dm_col} ~ C(Incentive) + C(date)  [freq_weights = _weight]")

    formula_twfe = f"{dm_col} ~ {treatment_expr} + {day_fe_expr}"
    twfe_model   = run_ols_clustered(formula_twfe, panel, cluster_s, weight_col="_weight")
    twfe_rows    = extract_incentive_coefs(twfe_model, treatment_s, ref)

    print(f"\n  n={int(twfe_model.nobs):,}  drivers={panel[driver_s].nunique():,}  "
          f"days={panel[date_s].nunique()}  "
          f"R²={twfe_model.rsquared:.3f}  Adj-R²={twfe_model.rsquared_adj:.3f}")
    print_coef_table(twfe_rows, ref, outcome_label)

    # ══════════════════════════════════════════════════
    # MODEL 2 — Pooled OLS (day FE only, no driver FE), weighted
    # ══════════════════════════════════════════════════
    print(f"\n{'─'*70}")
    print(f"MODEL 2: Pooled OLS  (day FE only, no driver FE, W4-normalised) — comparison")

    formula_pooled = f"{outcome_s} ~ {treatment_expr} + {day_fe_expr}"
    pooled_model   = run_ols_clustered(formula_pooled, panel, cluster_s, weight_col="_weight")
    pooled_rows    = extract_incentive_coefs(pooled_model, treatment_s, ref)

    print(f"\n  n={int(pooled_model.nobs):,}  "
          f"R²={pooled_model.rsquared:.3f}  Adj-R²={pooled_model.rsquared_adj:.3f}")
    print_coef_table(pooled_rows, ref, outcome_label)

    # ══════════════════════════════════════════════════
    # MODEL 3 — TWFE per calendar date, weighted
    #
    # For each incentive day d:
    #   sub-panel = rows for day d  +  all no-incentive days (baseline pool)
    #   apply W4 weights within the sub-panel before demeaning
    #   re-demean within that sub-panel (weighted driver FE)
    #   run: outcome_dm ~ C(Incentive) + C(date)  [freq_weights = _weight]
    #
    # The W4 weight is re-applied to the sub-panel so the per-day models
    # are consistent with Models 1 & 2.
    # ══════════════════════════════════════════════════
    print(f"\n{'─'*70}")
    print(f"MODEL 3: TWFE per calendar date  (day-by-day incentive effects, W4-normalised)")

    all_dates    = sorted(panel[date_s].unique())
    baseline_sub = panel[panel[treatment_s] == ref].copy()
    baseline_dates = set(baseline_sub[date_s].unique())

    inc_dates = [
        d for d in all_dates
        if (panel[panel[date_s] == d][treatment_s] != ref).any()
    ]

    print(f"  Total dates                  : {len(all_dates)}")
    print(f"  Pure baseline dates (W1+W4)  : {len(baseline_dates)}")
    print(f"  Dates with incentives        : {len(inc_dates)}")

    day_models   = {}
    all_day_rows = []

    for date in inc_dates:
        day_sub       = panel[panel[date_s] == date].copy()
        focal_drivers = set(day_sub[driver_s].unique())

        matched_baseline = baseline_sub[baseline_sub[driver_s].isin(focal_drivers)].copy()

        # _weight already set correctly from weight_baseline(); no re-assignment needed
        sub = pd.concat([day_sub, matched_baseline], ignore_index=True)

        # Weighted within-demeaning, consistent with the weight column
        sub, sub_dm = demean_within_driver(sub, outcome_s, driver_s, weight_col="_weight")

        if sub[date_s].nunique() < 2:
            continue

        try:
            formula_d = f"{sub_dm} ~ {treatment_expr} + C({date_s})"
            model_d   = run_ols_clustered(formula_d, sub, cluster_s, weight_col="_weight")

            date_label = str(date)
            day_models[date_label] = model_d

            rows = extract_incentive_coefs(model_d, treatment_s, ref)
            day_incentives = day_sub[treatment_s].unique()
            rows = [r for r in rows if r["incentive"] in day_incentives]

            for r in rows:
                r.update({
                    "date":      date_label,
                    "n_obs":     int(model_d.nobs),
                    "r2":        round(model_d.rsquared, 4),
                    "reference": ref,
                    "model":     "TWFE_by_day",
                })
            all_day_rows.extend(rows)

            print(f"\n  {date_label}  (n={int(model_d.nobs):,}  R²={model_d.rsquared:.3f})")
            print_coef_table(rows, ref, outcome_label)

        except Exception as e:
            print(f"\n  {date}: ERROR — {e}")

    # ══════════════════════════════════════════════════
    # MODEL 4 — TWFE × zone_type, weighted
    # ══════════════════════════════════════════════════
    zone_rows_all   = []
    marginal_df_out = None

    if zone_type_s and zone_type_s in panel.columns:
        print(f"\n{'─'*70}")
        print(f"MODEL 4: TWFE × Zone Type  (Win / Lose / Swing, W4-normalised)")
        print(f"  {dm_col} ~ C(Incentive) * C(zone_type) + C(date)  [freq_weights = _weight]")

        zone_types = sorted(panel[zone_type_s].dropna().unique())
        ref_zone   = panel[zone_type_s].value_counts().idxmax()
        zone_type_expr = f"C({zone_type_s}, Treatment('{ref_zone}'))"

        print(f"\n  Zone types: {zone_types}  |  Reference: '{ref_zone}'")

        formula_zone = (
            f"{dm_col} ~ "
            f"{treatment_expr} * {zone_type_expr} + "
            f"{day_fe_expr}"
        )

        try:
            zone_model = run_ols_clustered(formula_zone, panel, cluster_s, weight_col="_weight")

            print(f"\n  n={int(zone_model.nobs):,}  "
                  f"R²={zone_model.rsquared:.3f}  "
                  f"Adj-R²={zone_model.rsquared_adj:.3f}")

            zone_rows     = extract_interaction_coefs(zone_model, treatment_s, zone_type_s)
            zone_rows_all = zone_rows

            print(f"\n  Raw interaction coefficients:")
            print(f"  {'Term':<55} {'Coef':>9} {'SE':>7} {'p':>7}  Sig")
            print("  " + "─" * 85)
            for r in zone_rows:
                term_label = (f"{r['incentive']} × {r['zone_type']}"
                              if r["term"] == "interaction"
                              else f"{r['incentive']} [main / ref zone]")
                print(f"  {term_label:<55} {r['coef']:>9.4f} {r['se']:>7.4f}"
                      f" {r['pval']:>7.3f}  {sig(r['pval']):<3}")

            marginal_df     = compute_zone_marginal_effects(zone_rows, zone_types, ref_zone)
            marginal_df_out = marginal_df

            print(f"\n  Total marginal effects per zone:")
            print(f"  {'Incentive':<28} {'Zone':<10} {'Effect':>8}  Sig")
            print("  " + "─" * 55)
            for _, row in marginal_df.sort_values(["incentive","zone_type"]).iterrows():
                p = row.get("pval", np.nan)
                s = sig(p) if pd.notna(p) else ""
                print(f"  {row['incentive']:<28} {row['zone_type']:<10}"
                      f" {row['total_coef']:>8.4f}  {s}")

            print(f"\n  Pivot — total effect by incentive × zone:")
            pivot = marginal_df.pivot_table(
                index="incentive", columns="zone_type", values="total_coef"
            )
            print(pivot.round(4).to_string())

        except Exception as e:
            print(f"\n  Model 4 ERROR — {e}")
            import traceback; traceback.print_exc()

    # ══════════════════════════════════════════════════
    # SAVE
    # ══════════════════════════════════════════════════
    tag = safe(outcome_raw)
    os.makedirs(output_dir, exist_ok=True)

    for r in twfe_rows:
        r.update({"date": "TWFE_POOLED", "n_obs": int(twfe_model.nobs),
                   "r2": round(twfe_model.rsquared, 4),
                   "reference": ref, "model": "TWFE"})
    for r in pooled_rows:
        r.update({"date": "OLS_POOLED", "n_obs": int(pooled_model.nobs),
                   "r2": round(pooled_model.rsquared, 4),
                   "reference": ref, "model": "OLS_no_driverFE"})

    results_df = pd.DataFrame(twfe_rows + pooled_rows + all_day_rows)
    results_df = results_df[[
        "model","date","incentive","coef","se","tstat",
        "pval","ci_low","ci_high","n_obs","r2","reference"
    ]].round(4)

    csv1 = os.path.join(output_dir, f"did_day_results_{tag}.csv")
    results_df.to_csv(csv1, index=False)
    print(f"\n  ✓ {csv1}")

    day_rows_df = pd.DataFrame(all_day_rows)
    if not day_rows_df.empty:
        day_rows_df["date"] = pd.to_datetime(day_rows_df["date"], errors="coerce") \
                              .fillna(day_rows_df["date"])
        day_rows_df = day_rows_df.sort_values("date")
        day_rows_df["date"] = day_rows_df["date"].astype(str)

        pivot_coef = day_rows_df.pivot_table(
            index="incentive", columns="date", values="coef")
        pivot_pval = day_rows_df.pivot_table(
            index="incentive", columns="date", values="pval")

        twfe_s = pd.Series({r["incentive"]: r["coef"] for r in twfe_rows})
        pval_s = pd.Series({r["incentive"]: r["pval"] for r in twfe_rows})
        pivot_coef.insert(0, "TWFE_POOLED", twfe_s)
        pivot_pval.insert(0, "TWFE_POOLED", pval_s)

        csv2 = os.path.join(output_dir, f"did_day_pivot_{tag}.csv")
        with open(csv2, "w") as f:
            f.write(f"COEFFICIENTS — vs '{ref}' (driver FE + day FE, W4-normalised) — one column per date\n")
            pivot_coef.round(4).to_csv(f)
            f.write("\nP-VALUES\n")
            pivot_pval.round(4).to_csv(f)
        print(f"  ✓ {csv2}")

    if zone_rows_all:
        csv3 = os.path.join(output_dir, f"did_day_zone_{tag}.csv")
        pd.DataFrame(zone_rows_all).round(4).to_csv(csv3, index=False)
        print(f"  ✓ {csv3}")

    if marginal_df_out is not None:
        csv4 = os.path.join(output_dir, f"did_day_zone_marginal_{tag}.csv")
        marginal_df_out.round(4).to_csv(csv4, index=False)
        print(f"  ✓ {csv4}")

    try:
        all_models      = [twfe_model, pooled_model] + list(day_models.values())
        all_model_names = ["TWFE", "OLS"] + list(day_models.keys())
        incent_params   = [p for p in twfe_model.params.index if treatment_s in p]

        summary = summary_col(
            all_models, model_names=all_model_names,
            stars=True, float_format="%.4f",
            regressor_order=incent_params,
            info_dict={
                "N":  lambda m: f"{int(m.nobs):,}",
                "R²": lambda m: f"{m.rsquared:.3f}",
            },
        )
        txt = os.path.join(output_dir, f"did_day_summary_{tag}.txt")
        with open(txt, "w") as f:
            f.write(f"Day-Level TWFE DiD — Thailand Incentive Experiment\n")
            f.write(f"Outcome: {outcome_raw}\n")
            f.write(f"Reference: '{ref}' | SE clustered by {CLUSTER_BY}\n")
            f.write(f"Driver FE via weighted within-demeaning + Day FE via C(date)\n")
            f.write(f"W4 no-incentive rows weighted by {W1_DAYS}/{W4_DAYS} = {W4_WEIGHT:.4f} "
                    f"to equalise baseline week contributions\n\n")
            f.write(str(summary))
        print(f"  ✓ {txt}")
    except Exception as e:
        print(f"  (summary table skipped: {e})")

# ═══════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════

def main():
    print("=" * 70)
    print("Thailand Incentive Experiment — Day-Level DiD Panel Regression")
    print("Two-Way Fixed Effects: Driver FE + Day FE")
    print(f"Baseline normalisation: W4 rows weighted by {W1_DAYS}/{W4_DAYS} = {W4_WEIGHT:.4f}")
    print("=" * 70 + "\n")

    df = load_files(DATA_DIR)

    all_cols = OUTCOMES + [TREATMENT_COL, DRIVER_ID_COL, DATE_COL,
                           CLUSTER_BY, ZONE_TYPE_COL, "_week"]
    rename   = {c: safe(c) for c in all_cols if c and c in df.columns}
    df       = df.rename(columns=rename)

    treatment_s  = str(rename.get(TREATMENT_COL, safe(TREATMENT_COL)))
    driver_s     = str(rename.get(DRIVER_ID_COL,  safe(DRIVER_ID_COL)))
    date_s       = str(rename.get(DATE_COL,        safe(DATE_COL)))
    cluster_s    = str(rename.get(CLUSTER_BY,      safe(CLUSTER_BY)))    if CLUSTER_BY    else None
    zone_type_s  = str(rename.get(ZONE_TYPE_COL,   safe(ZONE_TYPE_COL))) if ZONE_TYPE_COL else None
    week_s       = "_week"

    if date_s not in df.columns:
        raise ValueError(
            f"Date column '{DATE_COL}' not found in data. "
            f"Available columns: {list(df.columns[:20])}"
        )

    ref = REFERENCE_LEVEL
    if ref not in df[treatment_s].values:
        ref = df[treatment_s].value_counts().idxmax()
        print(f"  '{REFERENCE_LEVEL}' not found — using '{ref}' as reference")
    print(f"  ✓ Reference: '{ref}'\n")

    print("Incentive × Week distribution (driver-day rows):")
    print(pd.crosstab(df[week_s], df[treatment_s]).to_string())

    if zone_type_s and zone_type_s in df.columns:
        print("\nZone type distribution:")
        print(df[zone_type_s].value_counts().to_string())
    else:
        print(f"\n  ⚠ '{ZONE_TYPE_COL}' not found — Model 4 will be skipped")
        zone_type_s = None

    for outcome_raw in OUTCOMES:
        outcome_s = safe(outcome_raw)
        if outcome_s not in df.columns:
            print(f"\n  ⚠ Outcome '{outcome_raw}' not found — skipping")
            continue

        df[outcome_s] = pd.to_numeric(df[outcome_s], errors="coerce")

        run_outcome(
            df           = df,
            outcome_raw  = outcome_raw,
            treatment_s  = treatment_s,
            driver_s     = driver_s,
            date_s       = date_s,
            cluster_s    = cluster_s,
            zone_type_s  = zone_type_s,
            week_s       = week_s,
            ref          = ref,
            output_dir   = OUTPUT_DIR,
        )

    print(f"\n{'='*70}")
    print("Done ✓")
    print("─" * 70)
    print("Outputs per outcome (completed, total_online_hour):")
    print("  did_day_results_{outcome}.csv        — full tidy results")
    print("  did_day_pivot_{outcome}.csv          — incentive × week matrix")
    print("  did_day_summary_{outcome}.txt        — paper-style table")
    print("  did_day_zone_{outcome}.csv           — zone interaction coefs")
    print("  did_day_zone_marginal_{outcome}.csv  — total effects per zone")
    print("─" * 70)
    print("How to read:")
    print(f"  coef = extra units per day vs '{ref}',")
    print("         controlling for driver baseline + day-level shocks")
    print("  W4 no-incentive rows weighted 4/14 to match W1 baseline length")
    print("  Day FE absorbs: day-of-week, demand spikes, weather, holidays")
    print("  *** p<0.01  ** p<0.05  * p<0.10")
    print("=" * 70)

if __name__ == "__main__":
    main()

Thailand Incentive Experiment — Day-Level DiD Panel Regression
Two-Way Fixed Effects: Driver FE + Day FE
Baseline normalisation: W4 rows weighted by 4/14 = 0.2857

  tiers_merged_all-0710-0713.csv                → W1_Jul10  (15,566 rows)
  tiers_merged_all-0714-0720.csv                → W2_Jul14  (29,301 rows)
  tiers_merged_all-0721-0727.csv                → W3_Jul21  (29,583 rows)
  tiers_merged_all-0728-0810.csv                → W4_Jul28  (63,437 rows)
  tiers_merged_all-0811-0815.csv                → W5_Aug11  (20,559 rows)
  tiers_merged_all-0816-0822.csv                → W6_Aug16  (30,336 rows)
  tiers_merged_all-0823-0829.csv                → W7_Aug23  (30,149 rows)

  Total rows: 218,931
  ✓ Reference: 'No Incentive'

Incentive × Week distribution (driver-day rows):
Incentive  Active Bonus  Daily Productivity  Guarantee  No Incentive  Streak
_week                                                                       
W1_Jul10              0                   0          0       

In [16]:
pd.set_option('display.max_rows', None)

In [31]:
pd.read_csv('./results/did_day_results_completed.csv')

,model,date,incentive,coef,se,tstat,pval,ci_low,ci_high,n_obs,r2,reference
0,TWFE,TWFE_POOLED,Active Bonus,-0.0461,0.0746,-0.6179,0.5366,-0.1924,0.1002,218931,0.0305,No Incentive
1,TWFE,TWFE_POOLED,Daily Productivity,-0.2012,0.0831,-2.4221,0.0154,-0.3640,-0.0384,218931,0.0305,No Incentive
2,TWFE,TWFE_POOLED,Guarantee,-2.3131,0.0871,-26.5711,0.0000,-2.4837,-2.1424,218931,0.0305,No Incentive
3,TWFE,TWFE_POOLED,Streak,-0.5773,0.0701,-8.2390,0.0000,-0.7146,-0.4400,218931,0.0305,No Incentive
4,OLS_no_driverFE,OLS_POOLED,Active Bonus,0.3810,0.1166,3.2668,0.0011,0.1524,0.6096,218931,0.0170,No Incentive
5,OLS_no_driverFE,OLS_POOLED,Daily Productivity,0.6556,0.1288,5.0906,0.0000,0.4032,0.9080,218931,0.0170,No Incentive
6,OLS_no_driverFE,OLS_POOLED,Guarantee,-2.5881,0.1207,-21.4450,0.0000,-2.8246,-2.3515,218931,0.0170,No Incentive
7,OLS_no_driverFE,OLS_POOLED,Streak,0.2714,0.1125,2.4124,0.0158,0.0509,0.4919,218931,0.0170,No Incentive
8,TWFE_by_day,2025-07-14,Active Bonus,-0.8254,0.3019,-2.7340,0.0063,-1.4171,-0.2337,81509,0.0308,No Incentive
9,TWFE_by_day,2025-07-14,Daily Productivity,-0.2865,0.3207,-0.8935,0.3716,-0.9151,0.3420,81509,0.0308,No Incentive


In [13]:
df = pd.read_csv('./results/did_day_results_completed.csv')
print(df['model'].value_counts())
print(df[df['model'] == 'TWFE_by_day'].head(10))


model
TWFE               4
OLS_no_driverFE    4
Name: count, dtype: int64
Empty DataFrame
Columns: [model, date, incentive, coef, se, tstat, pval, ci_low, ci_high, n_obs, r2, reference]
Index: []
